# 🏛️ SocialContract-v0 — GRPO Training Notebook

**Environment:** Fiscal Policy Advisory (OpenEnv-compliant)  
**Training Method:** GRPO (Group Relative Policy Optimization) via TRL  
**Efficient Fine-tuning:** Unsloth (4-bit LoRA)  
**Model:** Qwen2.5-1.5B-Instruct  

This notebook trains an LLM to be an IMF-grade economic policy advisor using
reinforcement learning from the SocialContract-v0 environment rewards.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth trl datasets transformers accelerate pydantic numpy matplotlib
!pip install --no-deps xformers
print('Dependencies installed!')

## 2. Clone the Environment

In [ ]:
# Clone the SocialContract-v0 environment repo
!git clone https://huggingface.co/spaces/Tyr-123/SocialContract-v0 /content/socialcontract
import sys
sys.path.insert(0, '/content/socialcontract')

# Verify environment works
from env.openenv_wrapper import SocialContractOpenEnv
from env.pydantic_models import PolicyAction
from graders.graders import run_grader

env = SocialContractOpenEnv('task1_stability')
obs = env.reset()
print(f'✅ Environment loaded! GDP={obs.gdp:.1f}, Gini={obs.gini:.3f}')

## 3. Quick Baseline Evaluation

In [ ]:
import numpy as np

# Random baseline
def run_random_episode(task_id, seed=42):
    rng = np.random.default_rng(seed)
    env = SocialContractOpenEnv(task_id, seed=seed)
    env.reset()
    while not env.is_done:
        action = PolicyAction(
            tax_delta=float(rng.uniform(-0.05, 0.05)),
            ubi_delta=float(rng.uniform(-5.0, 5.0)),
            public_good_delta=float(rng.uniform(-0.05, 0.05)),
            interest_rate_delta=float(rng.uniform(-0.02, 0.02)),
            stimulus_package=float(rng.uniform(0, 100)),
            import_tariff_delta=float(rng.uniform(-0.03, 0.03)),
            money_supply_delta=float(rng.uniform(-100, 100)),
            minimum_wage_delta=float(rng.uniform(-1, 1)),
            reasoning='random',
        )
        env.step(action)
    return run_grader(task_id, env._history)

tasks = ['task1_stability', 'task2_recession', 'task4_stagflation']
print('Random Baseline Scores:')
for task in tasks:
    grade = run_random_episode(task)
    print(f'  {task}: {grade["score"]:.4f} — {grade["verdict"]}')

## 4. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-1.5B-Instruct',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'✅ Model loaded: Qwen2.5-1.5B-Instruct with LoRA (r=16)')

## 5. Generate Training Prompts

In [ ]:
import random
from datasets import Dataset

SYSTEM_PROMPT = (
    'You are an expert economic policy advisor. You make precise, data-driven '
    'decisions to optimize economic outcomes across multiple competing objectives. '
    'You must respond ONLY with valid JSON matching the exact format requested. '
    'No markdown, no extra text — only the JSON object.'
)

def generate_prompts(n=80):
    prompts = []
    for task_id in tasks:
        for seed in range(1, n // len(tasks) + 2):
            if len(prompts) >= n:
                break
            env = SocialContractOpenEnv(task_id, seed=seed)
            obs = env.reset()
            warmup = random.randint(0, env.task_cfg['max_steps'] // 3)
            rng = np.random.default_rng(seed)
            for _ in range(warmup):
                action = PolicyAction(
                    tax_delta=float(rng.uniform(-0.02, 0.02)),
                    ubi_delta=float(rng.uniform(-1, 1)),
                    public_good_delta=float(rng.uniform(-0.02, 0.02)),
                    interest_rate_delta=float(rng.uniform(-0.01, 0.01)),
                    stimulus_package=float(rng.uniform(0, 30)),
                    import_tariff_delta=0.0,
                    money_supply_delta=float(rng.uniform(-30, 30)),
                    minimum_wage_delta=0.0,
                    reasoning='warmup',
                )
                obs, _, done, _ = env.step(action)
                if done: break
            if not env.is_done:
                prompts.append({
                    'messages': [
                        {'role': 'system', 'content': SYSTEM_PROMPT},
                        {'role': 'user', 'content': obs.to_prompt()},
                    ],
                    'task_id': task_id,
                    'seed': seed,
                    'warmup_steps': warmup,
                })
    random.shuffle(prompts)
    return prompts[:n]

prompt_data = generate_prompts(80)
dataset = Dataset.from_list(prompt_data)
print(f'✅ Generated {len(dataset)} training prompts')

## 6. Define Reward Function

In [ ]:
import re, json

def parse_action(text):
    try:
        if '```' in text:
            match = re.search(r'```(?:json)?\s*(.*?)```', text, re.DOTALL)
            if match: text = match.group(1)
        match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
        if match: text = match.group(0)
        parsed = json.loads(text)
        return PolicyAction(
            tax_delta=float(np.clip(parsed.get('tax_delta', 0), -0.1, 0.1)),
            ubi_delta=float(np.clip(parsed.get('ubi_delta', 0), -10, 10)),
            public_good_delta=float(np.clip(parsed.get('public_good_delta', 0), -0.1, 0.1)),
            interest_rate_delta=float(np.clip(parsed.get('interest_rate_delta', 0), -0.03, 0.03)),
            stimulus_package=float(np.clip(parsed.get('stimulus_package', 0), 0, 500)),
            import_tariff_delta=float(np.clip(parsed.get('import_tariff_delta', 0), -0.05, 0.05)),
            money_supply_delta=float(np.clip(parsed.get('money_supply_delta', 0), -500, 500)),
            minimum_wage_delta=float(np.clip(parsed.get('minimum_wage_delta', 0), -2, 2)),
            reasoning=str(parsed.get('reasoning', ''))[:200],
        )
    except:
        return None

def env_reward(completion, task_id, seed, warmup_steps):
    action = parse_action(completion)
    if action is None:
        return 0.0
    env = SocialContractOpenEnv(task_id, seed=seed)
    obs = env.reset()
    rng = np.random.default_rng(seed)
    for _ in range(warmup_steps):
        wa = PolicyAction(
            tax_delta=float(rng.uniform(-0.02, 0.02)),
            ubi_delta=float(rng.uniform(-1, 1)),
            public_good_delta=float(rng.uniform(-0.02, 0.02)),
            interest_rate_delta=float(rng.uniform(-0.01, 0.01)),
            stimulus_package=float(rng.uniform(0, 30)),
            import_tariff_delta=0.0,
            money_supply_delta=float(rng.uniform(-30, 30)),
            minimum_wage_delta=0.0,
            reasoning='warmup',
        )
        obs, _, done, _ = env.step(wa)
        if done: return 0.0
    obs, reward, done, _ = env.step(action)
    # Run remaining with a follow-through
    while not env.is_done:
        heur = PolicyAction(
            tax_delta=action.tax_delta * 0.5,
            ubi_delta=action.ubi_delta * 0.3,
            public_good_delta=action.public_good_delta * 0.5,
            interest_rate_delta=action.interest_rate_delta * 0.3,
            stimulus_package=0.0, import_tariff_delta=0.0,
            money_supply_delta=action.money_supply_delta * 0.2,
            minimum_wage_delta=action.minimum_wage_delta * 0.3,
            reasoning='followthrough',
        )
        obs, _, done, _ = env.step(heur)
    return run_grader(task_id, env._history)['score']

def reward_function(completions, **kwargs):
    rewards = []
    task_ids = kwargs.get('task_id', ['task1_stability'] * len(completions))
    seeds = kwargs.get('seed', [42] * len(completions))
    warmups = kwargs.get('warmup_steps', [0] * len(completions))
    for i, c in enumerate(completions):
        text = c[-1].get('content', '') if isinstance(c, list) else str(c)
        r = env_reward(text,
                       task_ids[i] if i < len(task_ids) else 'task1_stability',
                       seeds[i] if i < len(seeds) else 42,
                       warmups[i] if i < len(warmups) else 0)
        rewards.append(r)
    return rewards

print('✅ Reward function defined')

## 7. Train with GRPO

In [ ]:
from trl import GRPOTrainer, GRPOConfig

training_args = GRPOConfig(
    output_dir='./socialcontract-grpo',
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    max_completion_length=256,
    num_generations=4,
    logging_steps=5,
    save_steps=50,
    max_steps=150,
    report_to='none',
    bf16=torch.cuda.is_available(),
    temperature=0.7,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    reward_funcs=[reward_function],
)

print('🚀 Starting GRPO training...')
result = trainer.train()
print(f'✅ Training complete! Steps: {result.global_step}')

## 8. Post-Training Evaluation

In [ ]:
def evaluate_trained_model(task_id, seed=42):
    env = SocialContractOpenEnv(task_id, seed=seed)
    obs = env.reset()
    while not env.is_done:
        prompt = obs.to_prompt()
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, return_tensors='pt', add_generation_prompt=True
        ).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                inputs, max_new_tokens=256, temperature=0.1,
                do_sample=True, pad_token_id=tokenizer.eos_token_id,
            )
        resp = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        action = parse_action(resp)
        if action is None:
            action = PolicyAction(
                tax_delta=0.0, ubi_delta=0.0, public_good_delta=0.01,
                interest_rate_delta=0.0, stimulus_package=0.0,
                import_tariff_delta=0.0, money_supply_delta=0.0,
                minimum_wage_delta=0.0, reasoning='parse failure',
            )
        obs, _, _, _ = env.step(action)
    return run_grader(task_id, env._history)

print('Post-Training Scores:')
for task in tasks:
    grade = evaluate_trained_model(task)
    print(f'  {task}: {grade["score"]:.4f} — {grade["verdict"]}')

## 9. Save & Push to HuggingFace Hub

In [ ]:
# Save locally
model.save_pretrained('./socialcontract-grpo-final')
tokenizer.save_pretrained('./socialcontract-grpo-final')

# Optionally push to HuggingFace Hub
# from huggingface_hub import login
# login(token='your-hf-token')
# model.push_to_hub('your-username/socialcontract-policy-v0')
# tokenizer.push_to_hub('your-username/socialcontract-policy-v0')

print('✅ Model saved!')